# 23. End-to-End Case Study: Imbalanced Customer Transaction Fraud Detection

A complete classification pipeline for 1.8% rare fraud: spend ratios, velocity features, time-of-day risk, and PR-AUC optimization.


## 1. Objective
Build an end-to-end fraud detection pipeline for a severe 1.8% imbalanced dataset.
Key decisions:
1. Feature engineering for velocity and spending deviation.
2. Handling rare categories in merchant and device signatures.
3. Optimizing decision thresholds using Precision-Recall trade-offs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/fraud/transaction_fraud.csv')
print(f"Fraud Dataset: {df.shape[0]:,} rows | Positive Rate: {df['is_fraud'].mean():.2%}")


## 2. Feature Engineering: Velocity & Spend Ratios


In [ ]:
# 1. Parse timestamps
df['txn_dt'] = pd.to_datetime(df['transaction_time'])
df['hour'] = df['txn_dt'].dt.hour
df['is_night_txn'] = df['hour'].between(1, 5).astype(int)

# 2. Spend Deviation Ratios
df['spend_ratio'] = df['transaction_amount'] / (df['average_transaction_amount'] + 1e-5)
df['log_amount'] = np.log1p(df['transaction_amount'])

# 3. Account Age & Failed Attempts Interaction
df['risk_burst_index'] = df['failed_attempts'] * (df['transaction_count_24h'] + 1)
df['customer_age_imputed'] = df['customer_age'].fillna(df['customer_age'].median())
df['device_type_imputed'] = df['device_type'].fillna('Unknown')

# Feature matrices
cat_cols = ['merchant_category', 'device_type_imputed', 'location']
num_cols = ['transaction_amount', 'log_amount', 'spend_ratio', 'account_age_days', 
            'transaction_count_24h', 'previous_fraud_count', 'failed_attempts', 
            'risk_burst_index', 'is_night_txn', 'hour', 'customer_age_imputed']

X = df[cat_cols + num_cols]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")


## 3. Pipeline Construction & Model Training


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(cv=KFold(n_splits=5, shuffle=True, random_state=42), smooth="auto"), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', HistGradientBoostingClassifier(class_weight='balanced', random_state=42))
])

pipe.fit(X_train, y_train)
y_probs = pipe.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_probs)
pr_auc = average_precision_score(y_test, y_probs)

print("=" * 60)
print("FRAUD MODEL PERFORMANCE:")
print(f" - ROC-AUC:                  {roc_auc:.4f}")
print(f" - PR-AUC (Avg Precision):   {pr_auc:.4f}")
print("=" * 60)


## 4. Precision-Recall Curve & Threshold Tuning


In [ ]:
prec, recall, thresholds = precision_recall_curve(y_test, y_probs)

plt.figure(figsize=(10, 5))
plt.plot(recall, prec, color='#d95f02', lw=2.5, label=f'PR Curve (PR-AUC = {pr_auc:.4f})')
plt.axhline(df['is_fraud'].mean(), color='navy', linestyle='--', label=f"Baseline ({df['is_fraud'].mean():.1%})")
plt.title('Precision-Recall Curve for Rare Fraud Detection')
plt.xlabel('Recall (Fraud Detection Rate)')
plt.ylabel('Precision (Positive Predictive Value)')
plt.legend()
plt.tight_layout()
plt.show()
